## Imports and Setup

In [ ]:
import sys
sys.path.append('..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

from src import (
    NeuralThompson,
    FeaturePipeline,
    TREATMENTS, N_TREATMENTS, IDX_TO_TREATMENT, TREATMENT_TO_IDX,
    seed_everything, setup_plotting,
    TREATMENT_COLORS,
)
from src.data_generator import reward_oracle

seed_everything(42)
setup_plotting()
print("Setup complete")

## Define Handcrafted Patients

In [ ]:
## 20 patients designed with known clinically correct treatments.
## These represent distinct real-world clinical profiles.

patients = [
    # ── GLP-1 candidates: obese, NAFLD, weight-sensitive ──
    {
        "name": "Patient A1 — Young, Obese, NAFLD",
        "age": 42, "bmi": 38.5, "hba1c_baseline": 8.2, "egfr": 98.0,
        "diabetes_duration": 3.0, "fasting_glucose": 195.0, "c_peptide": 1.8,
        "bp_systolic": 138.0, "ldl": 142.0, "hdl": 38.0, "triglycerides": 220.0,
        "alt": 52.0, "cvd": 0, "ckd": 0, "nafld": 1, "hypertension": 1,
        "expected": "GLP-1",
        "clinical_reason": "Obese + NAFLD → GLP-1 reduces weight and liver fat",
    },
    {
        "name": "Patient A2 — Middle-aged, Severely Obese, NAFLD",
        "age": 50, "bmi": 42.0, "hba1c_baseline": 9.0, "egfr": 88.0,
        "diabetes_duration": 5.0, "fasting_glucose": 210.0, "c_peptide": 1.6,
        "bp_systolic": 140.0, "ldl": 130.0, "hdl": 35.0, "triglycerides": 250.0,
        "alt": 61.0, "cvd": 0, "ckd": 0, "nafld": 1, "hypertension": 1,
        "expected": "GLP-1",
        "clinical_reason": "Severe obesity + NAFLD + elevated ALT → GLP-1 first choice",
    },
    {
        "name": "Patient A3 — Obese, High Triglycerides, No CVD",
        "age": 47, "bmi": 36.0, "hba1c_baseline": 8.5, "egfr": 92.0,
        "diabetes_duration": 4.0, "fasting_glucose": 200.0, "c_peptide": 1.7,
        "bp_systolic": 135.0, "ldl": 125.0, "hdl": 36.0, "triglycerides": 280.0,
        "alt": 48.0, "cvd": 0, "ckd": 0, "nafld": 1, "hypertension": 0,
        "expected": "GLP-1",
        "clinical_reason": "Obesity + dyslipidaemia + NAFLD → GLP-1 for metabolic benefit",
    },
    {
        "name": "Patient A4 — Young Obese Female, Early Disease",
        "age": 38, "bmi": 39.0, "hba1c_baseline": 7.9, "egfr": 102.0,
        "diabetes_duration": 2.0, "fasting_glucose": 185.0, "c_peptide": 2.0,
        "bp_systolic": 130.0, "ldl": 118.0, "hdl": 40.0, "triglycerides": 195.0,
        "alt": 44.0, "cvd": 0, "ckd": 0, "nafld": 1, "hypertension": 0,
        "expected": "GLP-1",
        "clinical_reason": "Young + obese + preserved beta-cell → GLP-1 for weight loss",
    },

    # ── SGLT-2 candidates: CVD, heart failure risk ──
    {
        "name": "Patient B1 — CVD History",
        "age": 63, "bmi": 29.0, "hba1c_baseline": 8.8, "egfr": 72.0,
        "diabetes_duration": 9.0, "fasting_glucose": 215.0, "c_peptide": 1.2,
        "bp_systolic": 145.0, "ldl": 110.0, "hdl": 42.0, "triglycerides": 165.0,
        "alt": 28.0, "cvd": 1, "ckd": 0, "nafld": 0, "hypertension": 1,
        "expected": "SGLT-2",
        "clinical_reason": "CVD history → SGLT-2 reduces cardiovascular mortality",
    },
    {
        "name": "Patient B2 — Post-MI, Hypertension",
        "age": 67, "bmi": 30.5, "hba1c_baseline": 9.2, "egfr": 68.0,
        "diabetes_duration": 11.0, "fasting_glucose": 225.0, "c_peptide": 1.0,
        "bp_systolic": 150.0, "ldl": 105.0, "hdl": 40.0, "triglycerides": 180.0,
        "alt": 30.0, "cvd": 1, "ckd": 0, "nafld": 0, "hypertension": 1,
        "expected": "SGLT-2",
        "clinical_reason": "Post-MI + hypertension → SGLT-2 cardioprotective + BP lowering",
    },
    {
        "name": "Patient B3 — CVD, Overweight, Good Kidneys",
        "age": 60, "bmi": 31.0, "hba1c_baseline": 8.4, "egfr": 78.0,
        "diabetes_duration": 7.0, "fasting_glucose": 205.0, "c_peptide": 1.3,
        "bp_systolic": 142.0, "ldl": 115.0, "hdl": 44.0, "triglycerides": 158.0,
        "alt": 25.0, "cvd": 1, "ckd": 0, "nafld": 0, "hypertension": 1,
        "expected": "SGLT-2",
        "clinical_reason": "Established CVD + adequate eGFR → SGLT-2 proven mortality benefit",
    },
    {
        "name": "Patient B4 — Older CVD, Mild Overweight",
        "age": 70, "bmi": 28.0, "hba1c_baseline": 8.0, "egfr": 65.0,
        "diabetes_duration": 12.0, "fasting_glucose": 190.0, "c_peptide": 1.1,
        "bp_systolic": 148.0, "ldl": 108.0, "hdl": 45.0, "triglycerides": 155.0,
        "alt": 22.0, "cvd": 1, "ckd": 0, "nafld": 0, "hypertension": 1,
        "expected": "SGLT-2",
        "clinical_reason": "CVD dominant risk → SGLT-2 despite older age and mild eGFR drop",
    },

    # ── DPP-4 candidates: elderly, CKD, frail ──
    {
        "name": "Patient C1 — Elderly with CKD",
        "age": 74, "bmi": 26.5, "hba1c_baseline": 7.8, "egfr": 38.0,
        "diabetes_duration": 14.0, "fasting_glucose": 178.0, "c_peptide": 0.9,
        "bp_systolic": 142.0, "ldl": 98.0, "hdl": 52.0, "triglycerides": 140.0,
        "alt": 22.0, "cvd": 0, "ckd": 1, "nafld": 0, "hypertension": 1,
        "expected": "DPP-4",
        "clinical_reason": "Elderly + CKD → DPP-4 is renal-safe and well tolerated",
    },
    {
        "name": "Patient C2 — Very Elderly, Moderate CKD",
        "age": 79, "bmi": 25.0, "hba1c_baseline": 7.5, "egfr": 32.0,
        "diabetes_duration": 16.0, "fasting_glucose": 170.0, "c_peptide": 0.8,
        "bp_systolic": 138.0, "ldl": 92.0, "hdl": 55.0, "triglycerides": 130.0,
        "alt": 20.0, "cvd": 0, "ckd": 1, "nafld": 0, "hypertension": 1,
        "expected": "DPP-4",
        "clinical_reason": "Very elderly + moderate CKD → DPP-4 lowest hypoglycaemia risk",
    },
    {
        "name": "Patient C3 — Elderly, CKD, No CVD",
        "age": 72, "bmi": 27.0, "hba1c_baseline": 8.0, "egfr": 42.0,
        "diabetes_duration": 13.0, "fasting_glucose": 182.0, "c_peptide": 1.0,
        "bp_systolic": 140.0, "ldl": 100.0, "hdl": 50.0, "triglycerides": 145.0,
        "alt": 24.0, "cvd": 0, "ckd": 1, "nafld": 0, "hypertension": 0,
        "expected": "DPP-4",
        "clinical_reason": "Elderly + CKD stage 3 → DPP-4 dose-adjusted safely",
    },
    {
        "name": "Patient C4 — Elderly, Frail, Mild CKD",
        "age": 76, "bmi": 24.5, "hba1c_baseline": 7.6, "egfr": 45.0,
        "diabetes_duration": 15.0, "fasting_glucose": 172.0, "c_peptide": 0.85,
        "bp_systolic": 135.0, "ldl": 95.0, "hdl": 54.0, "triglycerides": 135.0,
        "alt": 21.0, "cvd": 0, "ckd": 1, "nafld": 0, "hypertension": 1,
        "expected": "DPP-4",
        "clinical_reason": "Frail elderly + CKD → DPP-4 safest option",
    },

    # ── Metformin candidates: early stage, lean, no comorbidities ──
    {
        "name": "Patient D1 — Early Stage, Lean",
        "age": 45, "bmi": 24.0, "hba1c_baseline": 7.4, "egfr": 95.0,
        "diabetes_duration": 1.5, "fasting_glucose": 155.0, "c_peptide": 1.9,
        "bp_systolic": 122.0, "ldl": 115.0, "hdl": 55.0, "triglycerides": 120.0,
        "alt": 18.0, "cvd": 0, "ckd": 0, "nafld": 0, "hypertension": 0,
        "expected": "Metformin",
        "clinical_reason": "Early stage, lean, no comorbidities → Metformin first-line",
    },
    {
        "name": "Patient D2 — Newly Diagnosed, Normal Weight",
        "age": 40, "bmi": 23.5, "hba1c_baseline": 7.2, "egfr": 100.0,
        "diabetes_duration": 0.5, "fasting_glucose": 148.0, "c_peptide": 2.1,
        "bp_systolic": 118.0, "ldl": 108.0, "hdl": 58.0, "triglycerides": 110.0,
        "alt": 16.0, "cvd": 0, "ckd": 0, "nafld": 0, "hypertension": 0,
        "expected": "Metformin",
        "clinical_reason": "Newly diagnosed + preserved function → Metformin standard of care",
    },
    {
        "name": "Patient D3 — Middle-aged, Mild Hyperglycaemia",
        "age": 52, "bmi": 25.5, "hba1c_baseline": 7.6, "egfr": 92.0,
        "diabetes_duration": 2.0, "fasting_glucose": 162.0, "c_peptide": 1.85,
        "bp_systolic": 125.0, "ldl": 120.0, "hdl": 52.0, "triglycerides": 125.0,
        "alt": 20.0, "cvd": 0, "ckd": 0, "nafld": 0, "hypertension": 0,
        "expected": "Metformin",
        "clinical_reason": "Mild disease + good kidneys + no comorbidities → Metformin",
    },
    {
        "name": "Patient D4 — Active, Lean, Early Disease",
        "age": 48, "bmi": 22.8, "hba1c_baseline": 7.3, "egfr": 98.0,
        "diabetes_duration": 1.0, "fasting_glucose": 150.0, "c_peptide": 2.0,
        "bp_systolic": 120.0, "ldl": 112.0, "hdl": 60.0, "triglycerides": 115.0,
        "alt": 17.0, "cvd": 0, "ckd": 0, "nafld": 0, "hypertension": 0,
        "expected": "Metformin",
        "clinical_reason": "Active lean patient + early T2D → Metformin with lifestyle",
    },

    # ── Insulin candidates: severe disease, beta-cell failure ──
    {
        "name": "Patient E1 — Severe, Advanced Disease",
        "age": 58, "bmi": 27.5, "hba1c_baseline": 11.5, "egfr": 62.0,
        "diabetes_duration": 18.0, "fasting_glucose": 285.0, "c_peptide": 0.3,
        "bp_systolic": 148.0, "ldl": 128.0, "hdl": 38.0, "triglycerides": 195.0,
        "alt": 35.0, "cvd": 1, "ckd": 1, "nafld": 0, "hypertension": 1,
        "expected": "Insulin",
        "clinical_reason": "Low C-peptide + severe HbA1c → beta-cell failure needs Insulin",
    },
    {
        "name": "Patient E2 — Very High HbA1c, Low C-peptide",
        "age": 55, "bmi": 26.0, "hba1c_baseline": 12.5, "egfr": 58.0,
        "diabetes_duration": 20.0, "fasting_glucose": 310.0, "c_peptide": 0.2,
        "bp_systolic": 152.0, "ldl": 135.0, "hdl": 35.0, "triglycerides": 210.0,
        "alt": 40.0, "cvd": 1, "ckd": 1, "nafld": 0, "hypertension": 1,
        "expected": "Insulin",
        "clinical_reason": "Severely depleted C-peptide + HbA1c>12 → Insulin mandatory",
    },
    {
        "name": "Patient E3 — Long Duration, Beta-cell Exhaustion",
        "age": 62, "bmi": 28.0, "hba1c_baseline": 11.0, "egfr": 55.0,
        "diabetes_duration": 22.0, "fasting_glucose": 270.0, "c_peptide": 0.25,
        "bp_systolic": 150.0, "ldl": 130.0, "hdl": 36.0, "triglycerides": 200.0,
        "alt": 38.0, "cvd": 1, "ckd": 1, "nafld": 0, "hypertension": 1,
        "expected": "Insulin",
        "clinical_reason": "22yr duration + near-zero C-peptide → beta-cell exhaustion",
    },
    {
        "name": "Patient E4 — Poorly Controlled, Multiple Comorbidities",
        "age": 60, "bmi": 29.0, "hba1c_baseline": 13.0, "egfr": 50.0,
        "diabetes_duration": 19.0, "fasting_glucose": 320.0, "c_peptide": 0.15,
        "bp_systolic": 155.0, "ldl": 140.0, "hdl": 33.0, "triglycerides": 225.0,
        "alt": 42.0, "cvd": 1, "ckd": 1, "nafld": 0, "hypertension": 1,
        "expected": "Insulin",
        "clinical_reason": "HbA1c>13 + CKD + CVD + minimal C-peptide → Insulin only option",
    },
]

print(f"Defined {len(patients)} handcrafted clinical profiles:")
for i, p in enumerate(patients):
    print(f"\n  Patient {i+1}: {p['name']}")
    print(f"    Expected:        {p['expected']}")
    print(f"    Clinical reason: {p['clinical_reason']}")

In [ ]:
print("Loading pipeline...")
pipe_scaled = FeaturePipeline.load("../models/feature_pipeline_scaled.joblib")
input_dim = len(pipe_scaled.features)

print("Loading NeuralThompson...")
model = NeuralThompson(input_dim=input_dim, hidden_dims=[128, 64], noise_variance=0.25)
model.load("../models/neural_thompson.pt")

print("NeuralThompson ready ✓")

## Predict Treatment for Each Patient

In [ ]:
print("=" * 60)
print("NEURALTHOMPSON — TREATMENT PREDICTIONS")
print("=" * 60)

predictions = []
for p in patients:
    x = pipe_scaled.transform_single(p)

    # Oracle ground truth
    oracle_rewards = [reward_oracle(p, t, noise=False) for t in TREATMENTS]
    oracle_action  = int(np.argmax(oracle_rewards))
    oracle_treat   = TREATMENTS[oracle_action]

    # NeuralThompson prediction
    action, _ = model.select_action(x)
    predicted  = IDX_TO_TREATMENT[action]
    correct    = predicted == oracle_treat

    predictions.append({
        "patient":   p["name"],
        "expected":  p["expected"],
        "oracle":    oracle_treat,
        "predicted": predicted,
        "correct":   correct,
        "rewards":   oracle_rewards,
        "clinical_reason": p["clinical_reason"],
    })

    print(f"\n{p['name']}")
    print(f"  Clinical reason:  {p['clinical_reason']}")
    print(f"  Oracle (truth):   {oracle_treat}")
    print(f"  NeuralThompson:   {predicted}  {'✓ CORRECT' if correct else '✗ INCORRECT'}")


## Accuracy Summary

In [ ]:
correct_count = sum(1 for r in predictions if r["correct"])
total = len(predictions)

print("\n" + "=" * 60)
print("NEURALTHOMPSON PREDICTION ACCURACY")
print("=" * 60)
print(f"\n  Correct predictions: {correct_count}/{total}")
print(f"  Accuracy:            {correct_count / total * 100:.0f}%")
print()
for r in predictions:
    status = "✓" if r["correct"] else "✗"
    print(f"  {status}  {r['patient']:<40} → {r['predicted']}")


plt.show()

## Reward Landscape Heatmap

In [ ]:
## Shows what the oracle reward function sees for each patient.
## NeuralThompson learns to approximate this landscape.

reward_matrix = np.array([r["rewards"] for r in predictions])
patient_labels = [f"P{i + 1}: {p['expected']}" for i, p in enumerate(patients)]

fig, ax = plt.subplots(figsize=(10, 5))
sns.heatmap(
    reward_matrix, annot=True, fmt=".2f", cmap="YlOrRd",
    xticklabels=TREATMENTS, yticklabels=patient_labels,
    ax=ax, linewidths=0.5,
)
ax.set_title(
    "Oracle Reward Landscape — Each Patient × Each Treatment\n"
    "Brightest cell = correct treatment. NeuralThompson learns this.",
    fontsize=11,
)
ax.set_xlabel("Treatment")
ax.set_ylabel("Patient")
plt.tight_layout()
plt.savefig("../results/nb08_reward_heatmap.png", dpi=150, bbox_inches='tight')

## Per-Patient Prediction Visualisation

In [ ]:
## Green bar = oracle best treatment
## Red bar   = NeuralThompson prediction
## Shows how close NeuralThompson is to the optimal choice.

fig, axes = plt.subplots(1, len(patients), figsize=(18, 5))

for i, (p, r) in enumerate(zip(patients, predictions)):
    ax = axes[i]
    bar_colors = []
    for t in TREATMENTS:
        if t == r["oracle"]:
            bar_colors.append("#4CAF50")   # green — oracle best
        elif t == r["predicted"]:
            bar_colors.append("#F44336")   # red — model pick
        else:
            bar_colors.append("#90A4AE")   # grey — other

    ax.bar(TREATMENTS, r["rewards"], color=bar_colors, edgecolor="white")
    status = "✓" if r["correct"] else "✗"
    ax.set_title(
        f"P{i+1} {status}\nOracle: {r['oracle']}\nModel: {r['predicted']}",
        fontsize=8,
    )
    ax.set_ylim(bottom=min(0, reward_matrix.min() - 0.5),
                top=reward_matrix.max() + 0.5)
    ax.tick_params(axis='x', rotation=45, labelsize=7)
    if i == 0:
        ax.set_ylabel("Expected Reward")

green_patch = mpatches.Patch(color="#4CAF50", label="Oracle best")
red_patch   = mpatches.Patch(color="#F44336", label="NeuralThompson pick")
grey_patch  = mpatches.Patch(color="#90A4AE", label="Other treatments")
fig.legend(handles=[green_patch, red_patch, grey_patch],
           loc="upper right", fontsize=9)

plt.suptitle(
    "NeuralThompson Predictions vs Oracle — Per Patient Reward Landscape",
    fontsize=12,
)
plt.tight_layout()
plt.savefig("../results/nb08_predictions.png", dpi=150, bbox_inches='tight')
plt.show()

## Context Sensitivity — Same Model, Different Patients

In [ ]:
## Proves NeuralThompson uses patient context to differentiate.
## If it ignored context, all patients would get the same treatment.

treatment_counts = {}
for r in predictions:
    t = r["predicted"]
    treatment_counts[t] = treatment_counts.get(t, 0) + 1

fig, ax = plt.subplots(figsize=(8, 4))
colors = [TREATMENT_COLORS[t] for t in TREATMENTS]
counts = [treatment_counts.get(t, 0) for t in TREATMENTS]
ax.bar(TREATMENTS, counts, color=colors, edgecolor='white')
ax.set_ylabel("Number of Patients Assigned")
ax.set_title(
    "NeuralThompson Treatment Distribution Across 5 Patients\n"
    "Variety proves the model uses patient context — not one-size-fits-all",
    fontsize=10,
)
for i, c in enumerate(counts):
    if c > 0:
        ax.text(i, c + 0.05, str(c), ha='center', fontsize=11)
ax.set_ylim(0, 3)
plt.tight_layout()
plt.savefig("../results/nb08_treatment_dist.png", dpi=150, bbox_inches='tight')
plt.show()

## Summary

In [ ]:
print("=" * 60)
print("NEURALTHOMPSON PREDICTION DEMO — SUMMARY")
print("=" * 60)
print(f"\n  Model:         NeuralThompson (winning contextual bandit)")
print(f"  Patients:      {total} handcrafted clinical profiles")
print(f"  Accuracy:      {correct_count}/{total} ({correct_count/total*100:.0f}%)")
print()
print("  How it works:")
print("    1. Patient clinical features fed into neural network")
print("    2. Network outputs reward estimate per treatment")
print("    3. Thompson sampling draws from uncertainty distribution")
print("    4. Treatment with highest sampled value is recommended")
print("    5. After observing outcome, posterior is updated")
print()
print("  Key insight: Each patient receives a DIFFERENT recommendation")
print("  based on their unique clinical profile — true personalisation.")
print("=" * 60)